# Social Media Sentiment Analysis
## Week 1: Dataset Collection & Text Preprocessing

---

- Learn Python and NLP basics
- Load a real-world dataset from Kaggle
- Clean and preprocess raw text data
- Perform Exploratory Data Analysis (EDA)
- Generate basic visualizations

---

**Dataset Used:** [Twitter Entity Sentiment Analysis — Kaggle](https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis)

This dataset contains real tweets labeled as Positive, Negative, Neutral, or Irrelevant. We use it as our social media data source.

In [ ]:
!pip install nltk textblob wordcloud matplotlib pandas numpy seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import warnings
warnings.filterwarnings('ignore')

for pkg in ['stopwords', 'punkt', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, quiet=True)

In [ ]:

 !pip install kaggle -q
 !kaggle datasets download -d jp797498e/twitter-entity-sentiment-analysis
 !unzip twitter-entity-sentiment-analysis.zip
 DATASET_PATH = 'twitter_training.csv'

try:
    df_raw = pd.read_csv(DATASET_PATH, header=None,
                         names=['id', 'entity', 'sentiment', 'text'])

    print(f'Shape: {df_raw.shape}')
except FileNotFoundError:
    print('Dataset file not found.')
    print('Please upload twitter_training.csv to this Colab session.')
    print()
    print('Using a smaller built-in sample for demonstration...')

    df_raw = pd.DataFrame(sample_data, columns=['id','entity','sentiment','text'])
    df_raw['id'] = range(1, len(df_raw)+1)
    print(f'Sample dataset created: {df_raw.shape}')

df_raw.head(10)

Dataset URL: https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis
License(s): CC0-1.0
twitter-entity-sentiment-analysis.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  twitter-entity-sentiment-analysis.zip
replace twitter_training.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:

print(f'Rows: {df_raw.shape[0]:,}  |  Columns: {df_raw.shape[1]}')
print()

print(df_raw.dtypes)


print(df_raw.isnull().sum())
print()

print(df_raw['sentiment'].value_counts())
print()

df_raw.head()

In [ ]:
df = df_raw[df_raw['sentiment'] != 'Irrelevant'].copy()
df = df.dropna(subset=['text']).reset_index(drop=True)
df['text'] = df['text'].astype(str)

print(f'Records after filtering: {len(df):,}')
print()
print('Sentiment counts after filtering:')
print(df['sentiment'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Dataset Overview', fontsize=16, fontweight='bold')

colors = {'Positive': '#2ecc71', 'Negative': '#e74c3c', 'Neutral': '#95a5a6'}
counts = df['sentiment'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=[colors.get(s, '#3498db') for s in counts.index],
            edgecolor='black', width=0.5)
for i, (label, val) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, val + counts.max()*0.01, str(val), ha='center',
                 fontweight='bold', fontsize=11)
axes[0].set_title('Sentiment Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment'); axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=[colors.get(s, '#3498db') for s in counts.index],
            startangle=140, explode=[0.04]*len(counts))
axes[1].set_title('Sentiment Share (%)', fontsize=13, fontweight='bold')

top_entities = df['entity'].value_counts().head(8)
axes[2].barh(top_entities.index[::-1], top_entities.values[::-1],
             color='#3498db', edgecolor='black')
axes[2].set_title('Top 8 Entities Mentioned', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Tweet Count')

plt.tight_layout()
plt.savefig('week1_dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: week1_dataset_overview.png')

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    text = text.lower()

    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    text = re.sub(r'@\w+', '', text)

    text = re.sub(r'#(\w+)', r'\1', text)

    text = text.encode('ascii', 'ignore').decode('ascii')

    text = re.sub(r'\d+', '', text)

    text = re.sub(r'[^a-z\s]', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    tokens = text.split()

    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return ' '.join(tokens)

df['cleaned_text'] = df['text'].apply(preprocess_text)

for i in range(5):
    print(f'\nOriginal : {df["text"].iloc[i]}')
    print(f'Cleaned  : {df["cleaned_text"].iloc[i]}')
    print('-'*65)
print(f'\nTotal records processed: {len(df):,}')

In [ ]:
df['original_length']  = df['text'].apply(len)
df['cleaned_length']   = df['cleaned_text'].apply(len)
df['word_count']       = df['cleaned_text'].apply(lambda x: len(x.split()))
df['char_reduction']   = ((df['original_length'] - df['cleaned_length']) / df['original_length'] * 100).round(1)

print(df[['original_length', 'cleaned_length', 'word_count', 'char_reduction']].describe().round(2))
print()
print(f'Average character reduction from cleaning: {df["char_reduction"].mean():.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Text Length Analysis', fontsize=14, fontweight='bold')

axes[0].hist(df['original_length'], bins=30, color='#3498db', edgecolor='black', alpha=0.8)
axes[0].set_title('Original Text Length'); axes[0].set_xlabel('Characters'); axes[0].set_ylabel('Frequency')

axes[1].hist(df['cleaned_length'], bins=30, color='#2ecc71', edgecolor='black', alpha=0.8)
axes[1].set_title('Cleaned Text Length'); axes[1].set_xlabel('Characters')

axes[2].hist(df['word_count'], bins=20, color='#e74c3c', edgecolor='black', alpha=0.8)
axes[2].set_title('Word Count per Post'); axes[2].set_xlabel('Words')

plt.tight_layout()
plt.savefig('week1_text_length_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def make_wordcloud(sentiment_label, colormap, ax, title):
    text = ' '.join(df[df['sentiment'] == sentiment_label]['cleaned_text'])
    if not text.strip():
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        return
    wc = WordCloud(width=700, height=380, background_color='white',
                   colormap=colormap, max_words=60, collocations=False).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Word Clouds by Sentiment', fontsize=15, fontweight='bold')

make_wordcloud('Positive', 'Greens',  axes[0], 'Positive Tweets')
make_wordcloud('Negative', 'Reds',    axes[1], 'Negative Tweets')
make_wordcloud('Neutral',  'Blues',   axes[2], 'Neutral Tweets')

plt.tight_layout()
plt.savefig('week1_wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from collections import Counter

def top_keywords(sentiment_label, n=12):
    words = ' '.join(df[df['sentiment'] == sentiment_label]['cleaned_text']).split()
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Top Keywords by Sentiment', fontsize=14, fontweight='bold')

for ax, label, color in zip(axes, ['Positive', 'Negative', 'Neutral'],
                                   ['#2ecc71', '#e74c3c', '#95a5a6']):
    kw = top_keywords(label)
    words_, counts = zip(*kw)
    ax.barh(list(words_)[::-1], list(counts)[::-1], color=color, edgecolor='black')
    ax.set_title(f'{label} Keywords', fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('week1_top_keywords.png', dpi=150, bbox_inches='tight')
plt.show()